In [1]:
"""
Imports
"""

import pandas as pd
import re
import torch
from tqdm import tqdm
from tqdm import trange
from torch import nn
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader, Dataset
from collections import Counter
from sklearn.model_selection import train_test_split

In [2]:
"""
Functions for dataset loading and preparation
"""

def load_dataset():
    df = pd.read_csv("dataset.csv", sep=";")
    return df

def tokenize(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)
    return text.split()

def build_vocab(texts, max_words=120):
    print("Building vocab...")
    counter = Counter()

    for text in texts:
        counter.update(tokenize(text))

    most_common = counter.most_common(max_words)
    vocab = {"<pad>": 0, "<unk>": 1}
    
    for i, (word, _) in enumerate(most_common, start=2):
        vocab[word] = i
    print("Finished building vocab!")
    return vocab

def encode(vocab, text, max_len=120):
    tokens = tokenize(text)
    ids = [vocab.get(token, vocab["<unk>"])
        for token in tokens]
    return ids[:max_len]

In [3]:
df = load_dataset()

In [4]:
"""
Splitting data into test and train
"""

# Shuffle data
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Divide datset into 80% train and 20% test
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['Label'])

print(f"Training samples: {len(train_df)}")
print("_________________")

print(f"Test samples: {len(test_df)}")
print("_________________")

print(f"\nTrain label distribution:\n{train_df['Label'].value_counts().sort_index()}")
print("_________________")

print(f"\nTest label distribution:\n{test_df['Label'].value_counts().sort_index()}")
print("_________________")

Training samples: 168244
_________________
Test samples: 42061
_________________

Train label distribution:
Label
Anthropic     8244
Google       40000
Human        40000
Meta         40000
OpenAI       40000
Name: count, dtype: int64
_________________

Test label distribution:
Label
Anthropic     2061
Google       10000
Human        10000
Meta         10000
OpenAI       10000
Name: count, dtype: int64
_________________


In [5]:
"""
Now we will build the vocab
"""

# A vocab of 50 000 + 2 (pad and unk)
# Maybe increase later...
vocab = build_vocab(train_df['Text'].values, max_words=50_000)
print(f"\nVocabulary size: {len(vocab)}")
print("_________________")

Building vocab...
Finished building vocab!

Vocabulary size: 50002
_________________


In [6]:
"""
Encode text and convert to tensor (for pytorch)
"""

class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab, label_to_idx, max_len=100):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.label_to_idx = label_to_idx
        self.max_len = max_len

    # Total number of samples in the dataset
    def __len__(self):
        return len(self.texts)

    
    def __getitem__(self, idx):
        # Convert the string label into a numerical index using label_to_idx
        text = self.texts[idx]
        label = self.label_to_idx[self.labels[idx]]

        # Encode the vocab
        encoded = encode(self.vocab, text, self.max_len)
        
        if len(encoded) < self.max_len:
            encoded += [0] * (self.max_len - len(encoded))
        
        return torch.tensor(encoded, dtype=torch.long), torch.tensor(label, dtype=torch.long)

In [7]:
"""
Setup dataset and dataloaders
"""

labels = sorted(df['Label'].unique()) # All label types

# Label mappings
label_to_idx = {label: i for i, label in enumerate(labels)}
idx_to_label = {i: label for label, i in label_to_idx.items()}

print(label_to_idx)

# Create pytorch object datasets
train_dataset = TextDataset(
    train_df['Text'].values,
    train_df['Label'].values,
    vocab,
    label_to_idx,
    max_len=100
)

test_dataset = TextDataset(
    test_df['Text'].values,
    test_df['Label'].values,
    vocab,
    label_to_idx,
    max_len=100
)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

{'Anthropic': 0, 'Google': 1, 'Human': 2, 'Meta': 3, 'OpenAI': 4}


In [8]:
print(f"\nTrain batches: {len(train_loader)}")
print("_________________")

print(f"Test batches: {len(test_loader)}")
print("_________________")


Train batches: 5258
_________________
Test batches: 1315
_________________


In [9]:
"""
Define a simple neural network model
Embedding + LSTM + Fully connected layers
"""

class TextClassifierNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, num_classes=5):
        super(TextClassifierNN, self).__init__()
        # Convert word indices into dense vectors
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        # Processes sequences of embeddings to capture context
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128), # Bidirectional (hidden_dim * 2)
            nn.ReLU(),
            nn.Dropout(0.4), # Regularization
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        output = self.fc(hidden)
        return output

In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {device}")

model = TextClassifierNN(vocab_size=len(vocab), embedding_dim=128, hidden_dim=256, num_classes=5)
model.to(device)

# Adam algorithm, should run hyperparameter optimization later...
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Weights to pay more attention to underrepresented classes
class_counts = train_df['Label'].value_counts().sort_index().values
weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
weights = weights / weights.sum()  # Normalize

# CrossEntropyLoss for multi-class classification
criterion = CrossEntropyLoss(weight=weights.to(device))


Using device: cpu


In [11]:
"""
Train and validation function
"""

# One epoch <=> One pass through all training data
def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, (texts, labels) in enumerate(dataloader):
        texts = texts.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad() # Clear gradient before loss.backward()
        outputs = model(texts)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        if (batch_idx + 1) % 100 == 0:
            print(f"\rBatch {batch_idx + 1}/{len(dataloader)}, Loss: {loss.item():.4f}", end="", flush=True)
            
    accuracy = 100 * correct / total
    avg_loss = total_loss / len(dataloader)
    return avg_loss, accuracy

# Perform one evaluation pass, without updating weights
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for texts, labels in dataloader:
            texts = texts.to(device)
            labels = labels.to(device)
            
            outputs = model(texts)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    avg_loss = total_loss / len(dataloader)
    return avg_loss, accuracy

In [ ]:
"""
Train Model with early stopping
"""
 
#best_test_acc = 0
best_test_loss = float('inf')
patience = 2
patience_counter = 0
num_epochs = 10

for epoch in range(num_epochs):
    print(f"Training Epoch {epoch+1}/{num_epochs}...")
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    test_loss, test_acc = validate(model, test_loader, criterion, device)
    
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.2f}%")
    print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.2f}%")
    print("-" * 60)
    
    # Early stopping loss instead of accuracy
    """
    if test_acc > best_test_acc:
        best_test_acc = test_acc
        patience_counter = 0
    """
    if test_loss < best_test_loss:
        best_test_loss = test_loss
        patience_counter = 0
        # Save best model
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"✓ New best model saved! Test Accuracy: {test_acc:.2f}%")
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{patience}")
        
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            # Load best model
            model.load_state_dict(torch.load('best_model.pth'))
            break

print(f"\nFinal Best Test Accuracy: {best_test_loss:.2f}%")

In [12]:
# Prediction function
def predict(text, model, vocab, device, idx_to_label):
    model.eval()
    with torch.no_grad():
        encoded = encode(vocab, text, max_len=100)
        if len(encoded) < 100:
            encoded += [0] * (100 - len(encoded))
        
        input_tensor = torch.tensor([encoded], dtype=torch.long).to(device)
        output = model(input_tensor)
        probabilities = torch.softmax(output, dim=1)
        prediction = torch.argmax(probabilities, dim=1).item()
        confidence = probabilities[0][prediction].item()
    
    source = idx_to_label[prediction]
    return source, confidence

In [13]:
"""
Predict dataset_exemplos
"""
model.load_state_dict(torch.load('best_model.pth'))

exemplos_df = pd.read_csv("dataset-exemplos.csv", sep=";")
real_res = exemplos_df["Label"].tolist()

predicted_res = []
for text in exemplos_df["Text"]:
    source, _ = predict(text, model, vocab, device, idx_to_label)
    predicted_res.append(source)

# Calculate accuracy
correct = sum(1 for r, p in zip(real_res, predicted_res) if r == p)
accuracy = correct / len(real_res)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.4000


In [15]:
"""
Predict submission and save as csv
"""
subm1_df = pd.read_csv("subm1.csv", sep=";")

results = []
for text in subm1_df["Text"]:
    source, _ = predict(text, model, vocab, device, idx_to_label)
    results.append(source)

subm1_df["Label"] = results

subm1_df.to_csv("subm1-g1-MMC-B.csv", index=False)
